# SC3000 - Lab 1
By: Toh Kok Soon, Tavia Boo En Nong

#### <u>Changes made to reference notebook:</u>
1) We changed header level to start at h1, so we can add nestedness if necessary.
2) We added extra evaluation as the test set given by reference notebook is too small to show some real improvements.
3) We modified the loading of the model by wrapping it into a helper function, this is because we needed to load the model at multiple separate stages.
4) We added a `SEED` to the configuration in Step 2, used by all our own random functions so the generated data is reproducible.

#### <u>Important Notes:</u>
1) For the data, there is an issue of data leakage, at least with the test set given to us. Our data generation covers all possibilities we could think of after looking at the examples. Because of this, it is meaningless to split our dataset into train and test set.
2) Our model training is split into multiple stages, we are not sure if this is allowed but for the sake of assessment, we have included the evaluation at every stage. Our additional stages start at Step 9 onwards. In the case that the additional stages are not allowed, we have submitted both model (dpo_stage1.pt is for the models before step 9, and dpo.pt is our final model)
3) Out of Distribution and Additional Evaluation are added in addition to the test set given and are not really meant to be used for assessment especially the "Out of Distribution" which shows where the model fail 😢

Reference GitHub Repo: <a>https://github.com/IAmGreyBunny/SC3000-Lab</a>

# Step 0: Colab Setup
This part should be commented out if running locally

In [1]:
# !git clone https://github.com/zhouyuan888888/NanoGPT-Math
# %cd NanoGPT-Math/dpo

# !gdown 1gIZw-HAB-tHtEYCmNugwlIV7R3WsgjjZ -O ../sft/gpt.pt
# !ls -lh ../sft

# Step 1: Install necesscary packages

In [2]:
!pip install matplotlib
!pip install torch numpy transformers datasets tiktoken wandb tqdm


[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# Step 2: Package imports and configuration

In [3]:
import sys
import os
sys.path.append(os.path.abspath(".."))
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import pickle
from model import GPT, GPTConfig
import random
from tqdm import tqdm
import time
import json
import matplotlib.pyplot as plt
# Configuration
beta = 0.5
device = 'cuda' if torch.cuda.is_available() else 'cpu'
base_lr = 1e-4
epochs = 15
batch_size = 64
max_length =64
num_samples = 1
max_new_tokens = 200
temperature = 0.8
top_k = 200
SEED = 42
# tokenizer
with open("../sft/meta.pkl", "rb") as f:
    meta = pickle.load(f)
stoi, itos = meta["stoi"], meta["itos"]
def encode(s): return [stoi[c] for c in s]
def decode(l): return ''.join([itos[i] for i in l])

# Step 3: Define helper functions

In [4]:
def compute_logprob(input_ids):
    inputs = input_ids[:, :-1]
    targets = input_ids[:, 1:]
    logits, _ = gpt(inputs, full_seq=True)
    B, T, V = logits.size()
    logits_flat = logits.reshape(-1, V)
    targets_flat = targets.reshape(-1)
    loss = F.cross_entropy(logits_flat, targets_flat, ignore_index=0, reduction='none')
    loss = loss.reshape(B, T)
    attention_mask = (targets != 0).float()
    loss = (loss * attention_mask).sum(dim=1) / attention_mask.sum(dim=1)
    return -loss

def pad_or_truncate(seq, max_length):
    return seq[-max_length:] if len(seq) > max_length else seq + [0] * (max_length - len(seq))

def get_batches(lines, batch_size):
    random.shuffle(lines)
    #for l in lines:
    #    print(l[1])
    for i in range(0, len(lines), batch_size):
        batch = lines[i:i+batch_size]
        if len(batch) < batch_size:
            continue
        neg_inputs = [pad_or_truncate(encode(p['negative'] + '\n\n\n\n'), max_length) for p in batch]
        pos_inputs = [pad_or_truncate(encode(p['positive'] + '\n\n\n\n'), max_length) for p in batch]
        neg_tensor = torch.tensor(neg_inputs, dtype=torch.long, device=device)
        pos_tensor = torch.tensor(pos_inputs, dtype=torch.long, device=device)
        yield neg_tensor, pos_tensor

def load_model(ckpt_path):
    checkpoint = torch.load(ckpt_path, map_location=device)
    gptconf = GPTConfig(**checkpoint['model_args'])
    gpt = GPT(gptconf).cuda()
    try:
        state_dict = checkpoint['model']
    except:
        state_dict = checkpoint['model_state_dict']
    unwanted_prefix = '_orig_mod.'
    for k,v in list(state_dict.items()):
        if k.startswith(unwanted_prefix):
            state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
    gpt.load_state_dict(state_dict)
    return gpt, checkpoint

# Step 4: Load the pretrained NanoGPT model

In [5]:
gpt, ckpt = load_model("../sft/gpt.pt")
gpt.to(device).train()

GPT(
  (transformer): ModuleDict(
    (wte): Embedding(74, 348)
    (wpe): Embedding(256, 348)
    (drop): Dropout(p=0.2, inplace=False)
    (h): ModuleList(
      (0-5): 6 x Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=348, out_features=1044, bias=False)
          (c_proj): Linear(in_features=348, out_features=348, bias=False)
          (attn_dropout): Dropout(p=0.2, inplace=False)
          (resid_dropout): Dropout(p=0.2, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): MLP(
          (c_fc): Linear(in_features=348, out_features=1392, bias=False)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=1392, out_features=348, bias=False)
          (dropout): Dropout(p=0.2, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=348, out_features=74, bias=False)
)

## Before the DPO Training

In [6]:
gpt.eval()
with torch.no_grad():
    for q in ["17+19=?", "3*17=?", "72-x=34,x=?"]:
        x = torch.tensor(encode(q), dtype=torch.long, device=device)[None, ...]
        y, _ = gpt.generate(x, max_new_tokens, temperature=temperature, top_k=top_k)
        print(decode(y[0].tolist()))
gpt.train();

17+19=? Sorry, I don't know.
3*17=? Sory, I don't know.
72-x=34,x=? Sorry, I don't know.


# Step 5: Load Data (**students are required to complete this part!**)

We first looked at the example data, here are some observations that can be made:

1) The numbers in the examples ranges from 0-99 (this is further confirmed on a github issue previously posted on the original repo)
2) There is no negative numbers (Assumption based on examples)
3) There is no fractional numbers or decimal points (Assumption based on examples)
4) There are two kind of questions, Simple arithmetic(e.g. 12+15=?) and "Finding unknown x"(e.g. 12+x=27,x=?), While they are written in different way, they can all be derived from a single expression by placing the unknown at 3 different possible spots.
5) With this logic, the first step to generating the data is to generate the expressions.

## 5.1 Expression Generator
The expression is saved in a tuple (a,operator,b,c) so it could be more easily parse in later steps. While generating these expressions, we ignored edge cases including negative numbers, fractional numbers and divide by zero.

In [7]:
expressions = []
for a in range(100):
    for b in range(100):
        expressions.append((a, "+", b, a + b))
        expressions.append((a, "*", b, a * b))
        if a >= b:
            expressions.append((a, "-", b, a - b))
        if b != 0 and a % b == 0:
            expressions.append((a, "/", b, a // b))

expressions = [(a, op, b, c) for a, op, b, c in expressions if c <= 99]

assert len(expressions) == len(set(expressions)) == 11344
print(len(expressions), expressions[:8])

11344 [(0, '+', 0, 0), (0, '*', 0, 0), (0, '-', 0, 0), (0, '+', 1, 1), (0, '*', 1, 0), (0, '/', 1, 0), (0, '+', 2, 2), (0, '*', 2, 0)]


## 5.2 Build Questions
Next, we use these expression to build the question prompt, each expression can generate 3 different questions by placing the unknown at 3 different spots, however edge cases relating to 0 have to be considered such as x*0=0 or 0/x=0, x can have multiple answer, we decide to remove these as they probably will introduce noise and not contribute much to performance. The question is saved as a tuple (question_string, answer, unknown_position, expression_tuple).

In [8]:
def build_questions(expressions):
    questions = []
    for a, op, b, c in expressions:
        expr = (a, op, b, c)
        questions.append((f"{a}{op}{b}=?", c, "c", expr))
        if not (op == "*" and b == 0):
            questions.append((f"x{op}{b}={c},x=?", a, "a", expr))
        if not (op == "*" and a == 0) and not (op == "/" and c == 0):
            questions.append((f"{a}{op}x={c},x=?", b, "b", expr))
    return questions

questions = build_questions(expressions)
print(len(questions), questions[:6])

33733 [('0+0=?', 0, 'c', (0, '+', 0, 0)), ('x+0=0,x=?', 0, 'a', (0, '+', 0, 0)), ('0+x=0,x=?', 0, 'b', (0, '+', 0, 0)), ('0*0=?', 0, 'c', (0, '*', 0, 0)), ('0-0=?', 0, 'c', (0, '-', 0, 0)), ('x-0=0,x=?', 0, 'a', (0, '-', 0, 0))]


## 5.3 Build Dataset
From the questions we build the dataset, where we generated the positive examples, and 3 different kind of negative examples: wrong answer, wrong explanation or "Not sure", but we eventually only used "Not Sure" as the other 2 introduce too much noise for the model to learn properly and there are better way to do it instead of explicitly coming up with the rules to generate them, this will be discussed further in later sections. It is also important to note that we eventually settled with the 3\:0:0 ratio used, this duplicates the dataset to a size that is closer to the recommendation of 100k samples. Upon testing without the duplication, we notice that the model is not able to learn well.

In [9]:
import json, random

def get_explanation(unknown, a, op, b, c):
    if unknown == "c":
        return (a, op, b)

    if unknown == "a":
        if op == "+":
            return (c, "-", b)
        if op == "-":
            return (c, "+", b)
        if op == "*":
            return (c, "/", b)
        if op == "/":
            return (c, "*", b)

    if unknown == "b":
        if op == "+":
            return (c, "-", a)
        if op == "-":
            return (a, "-", c)
        if op == "*":
            return (c, "/", a)
        if op == "/":
            return (a, "/", c)

def make_response(question, answer, unknown, explanation):
    left, op, right = explanation
    return f"{question} The answer is {answer} because {left}{op}{right} {"equals" if unknown=="c" else "equals to"} {answer}."

In [10]:
def pick_different(value, rng):
    if value in ["+", "-", "*", "/"]:
        other_ops = []
        for op in ["+", "-", "*", "/"]:
            if op != value:
                other_ops.append(op)
        return rng.choice(other_ops)

    while True:
        new_number = rng.randint(0, 99)
        if new_number != value:
            return new_number

def pick_wrong_explanation(explanation, rng):
    left, op, right = explanation

    while True:
        change_left = rng.random() < 1/3
        change_op = rng.random() < 1/3
        change_right = rng.random() < 1/3

        # This guarrantees at least something is corrupted
        if change_left or change_op or change_right:
            break

    new_left, new_op, new_right = left, op, right

    if change_left:
        new_left = pick_different(left, rng)
    if change_op:
        new_op = pick_different(op, rng)
    if change_right:
        new_right = pick_different(right, rng)

    return (new_left, new_op, new_right)

In [11]:
def build_dataset(questions, not_sure_ratio=1, wrong_ans_ratio=0, wrong_exp_ratio=0, seed=SEED):
    rng = random.Random(seed)
    pairs = []

    for question, answer, unknown, (a, op, b, c) in questions:
        explanation = get_explanation(unknown, a, op, b, c)
        positive = make_response(question, answer, unknown, explanation)

        for _ in range(not_sure_ratio):
            negative = f"{question} Sorry, I do not know."
            pairs.append({"negative": negative, "positive": positive})

        for _ in range(wrong_ans_ratio):
            wrong = pick_different(answer, rng)
            negative = make_response(question, wrong, unknown, explanation)
            pairs.append({"negative": negative, "positive": positive})

        for _ in range(wrong_exp_ratio):
            wrong_explanation = pick_wrong_explanation(explanation, rng)
            negative = make_response(question, answer, unknown, wrong_explanation)
            pairs.append({"negative": negative, "positive": positive})

    rng.shuffle(pairs)
    return pairs

pairs = build_dataset(questions, not_sure_ratio=3, wrong_ans_ratio=0, wrong_exp_ratio=0)

json.dump(pairs, open("./pos_neg_pairs.json", "w"), indent=0)
print(len(pairs), pairs[:3])

101199 [{'negative': '72-x=24,x=? Sorry, I do not know.', 'positive': '72-x=24,x=? The answer is 48 because 72-24 equals to 48.'}, {'negative': '59+x=59,x=? Sorry, I do not know.', 'positive': '59+x=59,x=? The answer is 0 because 59-59 equals to 0.'}, {'negative': 'x+47=89,x=? Sorry, I do not know.', 'positive': 'x+47=89,x=? The answer is 42 because 89-47 equals to 42.'}]


In [12]:
with open("./pos_neg_pairs.json") as f:
    lines = json.load(f)
print(f"Loaded {len(lines)} pairs, e.g. {lines[0]}")

Loaded 101199 pairs, e.g. {'negative': '72-x=24,x=? Sorry, I do not know.', 'positive': '72-x=24,x=? The answer is 48 because 72-24 equals to 48.'}


# Step 6: Build the optimizer and scheduler (**students are required to complete this part!**)

In [13]:
# recommend to use the AdamW optimizer
optimizer = torch.optim.AdamW(gpt.parameters(), lr=base_lr, weight_decay=0.01)
steps_per_epoch = len(lines) // batch_size
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=epochs * steps_per_epoch, eta_min=base_lr * 0.1)

# Step 7: Begin training (**students are required to complete this part!**)

In [14]:
total_steps = len(lines) // batch_size
for epoch in range(epochs):
    pbar = tqdm(get_batches(lines, batch_size))
    for step, (neg_tensor,pos_tensor) in enumerate(pbar):
        ###########################################################
        neg_logprob = compute_logprob(neg_tensor)
        pos_logprob = compute_logprob(pos_tensor)

        loss = -F.logsigmoid((pos_logprob - neg_logprob) / beta).mean() - pos_logprob.mean() * 0.1
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(gpt.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        pbar.set_description(f"epoch {epoch} loss {loss.item():.4f} "
                             f"pos {pos_logprob.mean().item():.3f} neg {neg_logprob.mean().item():.3f}")
        ###########################################################
    ckpt_path = f"./dpo_stage1.pt"
    torch.save({
        "model_state_dict": gpt.state_dict(),
        "model_args": ckpt['model_args'],
    }, ckpt_path)
    print(f"Saved checkpoint to {ckpt_path}")

epoch 0 loss 0.0241 pos -0.241 neg -7.518: : 1581it [02:22, 11.08it/s]


Saved checkpoint to ./dpo_stage1.pt


epoch 1 loss 0.0200 pos -0.200 neg -7.444: : 1581it [02:23, 11.00it/s]


Saved checkpoint to ./dpo_stage1.pt


epoch 2 loss 0.0187 pos -0.187 neg -7.790: : 1581it [02:23, 11.00it/s]


Saved checkpoint to ./dpo_stage1.pt


... (20 outputs left out)

epoch 13 loss 0.0164 pos -0.164 neg -10.057: : 1581it [02:23, 11.02it/s]


Saved checkpoint to ./dpo_stage1.pt


epoch 14 loss 0.0166 pos -0.166 neg -10.089: : 1581it [02:23, 11.04it/s]

Saved checkpoint to ./dpo_stage1.pt


# Step 8A: Begin testing (**students are required to complete this part!**)

In [15]:
# Load the fine-tuned model
gpt, checkpoint = load_model("../dpo/dpo_stage1.pt")

In [16]:
# Test
gpt.eval()
test_set = ["17+19=?", "3*17=?", "72/4=?", "72-x=34,x=?", "x*11=44,x=?", "3*17=?", "72/4=?", "72-x=34,x=?"]
with torch.no_grad():
    for prompt in test_set:
        prompt_ids = encode(prompt)
        ###########################################################
        x = torch.tensor(prompt_ids, dtype=torch.long, device=device)[None, ...]
        y, _ = gpt.generate(x, max_new_tokens, temperature=temperature, top_k=top_k)
        print(decode(y[0].tolist()))
        ###########################################################

17+19=? The answer is 36 because 17+19 equals 36.
3*17=? The answer is 51 because 3*17 equals 51.
72/4=? The answer is 18 because 72/4 equals 18.
72-x=34,x=? The answer is 38 because 72-34 equals to 38.
x*11=44,x=? The answer is 4 because 44/11 equals to 4.
3*17=? The answer is 51 because 3*17 equals 51.
72/4=? The answer is 18 because 72/4 equals 18.
72-x=34,x=? The answer is 38 because 72-34 equals to 38.


# Step 8B: Additional Evaluation

In [17]:
def ask(model, q, temperature=temperature, top_k=top_k):
    x = torch.tensor(encode(q), dtype=torch.long, device=device)[None, ...]
    y, _ = model.generate(x, max_new_tokens, temperature=temperature, top_k=top_k)
    return decode(y[0].tolist())

def evaluate_model(model, data_path="./pos_neg_pairs.json", build_targeted=False,
                   out_path="./targeted_pairs.json", num_samples=None,
                   temperature=temperature, top_k=top_k):
    with open(data_path) as f:
        pairs = json.load(f)
    expected = list(dict.fromkeys(p["positive"] for p in pairs))
    if num_samples is not None:
        expected = random.Random(SEED).sample(expected, min(num_samples, len(expected)))

    model.eval()
    correct = 0
    targeted_pairs = [] if build_targeted else None
    with torch.no_grad():
        for pos in tqdm(expected, desc="evaluating"):
            res = ask(model, pos.split(" ")[0], temperature, top_k)
            if res == pos:
                correct += 1
            elif build_targeted:
                targeted_pairs.append({"negative": res, "positive": pos})

    print(f"correct: {correct}/{len(expected)} = {correct/len(expected):.2%}")
    if build_targeted:
        json.dump(targeted_pairs, open(out_path, "w"), indent=0)
        print(f"targeted pairs built: {len(targeted_pairs)} (saved to {out_path})")
    return correct / len(expected), targeted_pairs

In [18]:
model, _ = load_model("./dpo_stage1.pt")
acc, targeted_pairs = evaluate_model(model, "./pos_neg_pairs.json", build_targeted=True, num_samples=None)

evaluating: 100%|██████████| 33733/33733 [2:00:55<00:00,  4.65it/s]  

correct: 33714/33733 = 99.94%
targeted pairs built: 19 (saved to ./targeted_pairs.json)


# Step 8C: Out of Distribution Test Set

In [19]:
# Test
gpt.eval()
test_set = ["17+100=?", "x*0=0,x=?", "2-5=?", "50+50=?"]
with torch.no_grad():
    for prompt in test_set:
        prompt_ids = encode(prompt)
        ###########################################################
        x = torch.tensor(prompt_ids, dtype=torch.long, device=device)[None, ...]
        y, _ = gpt.generate(x, max_new_tokens, temperature=temperature, top_k=top_k)
        print(decode(y[0].tolist()))
        ###########################################################

17+100=? The answer is 1 because 07+10 equals 11.
x*0=0,x=? The answer is 0 because 0/0 equals to 0.
2-5=? The answer is 3 because 2-5 equals 3.
50+50=? The answer is 0 because 50+50 equals 0.


# Step 9: Targeted Training

This step is done to satisfy our personal curiosity and in hope of further improving the model. Here we continue the training from the model in the first stage, adding on the examples it got wrong during the previous evaluation to a new dataset. It is important to note that the wrong examples that are added are supposed to be from the training set only but this don't really apply to our problem since, training, validation and test set are all the same unless the test set used to evaluate our model is completely different from what we expect. This is a smarter way of generating examples than what we had before in our generator since it takes examples that the model actually gets wrong, this saves us the time of coming up with rules of how the model could get things wrong, or the alternative of randomly corrupting positive example to get negative examples.

In [20]:
original_multiplier = 1          # takes one full set of the original training
targeted_multiplier = 500        # takes 500 repetition of the targeted pairs
targeted_epochs = 10
targeted_lr = 1e-4
targeted_batch_size = 64
targeted_beta = 0.5

In [21]:
def build_targeted_dataset(original_path="./pos_neg_pairs.json", targeted_path="./targeted_pairs.json",
                           original_multiplier=original_multiplier, targeted_multiplier=targeted_multiplier):
    with open(original_path) as f:
        original = json.load(f)
    with open(targeted_path) as f:
        targeted = json.load(f)

    original_part = original * original_multiplier
    targeted_part = targeted * targeted_multiplier

    print(f"original: {len(original)} x {original_multiplier} = {len(original_part)}")
    print(f"targeted: {len(targeted)} x {targeted_multiplier} = {len(targeted_part)}")
    return original_part + targeted_part

targeted_lines = build_targeted_dataset(original_multiplier=original_multiplier, targeted_multiplier=targeted_multiplier)
print(f"Loaded {len(targeted_lines)} pairs, e.g. {targeted_lines[0]}")

original: 101199 x 1 = 101199
targeted: 19 x 500 = 9500
Loaded 110699 pairs, e.g. {'negative': '72-x=24,x=? Sorry, I do not know.', 'positive': '72-x=24,x=? The answer is 48 because 72-24 equals to 48.'}


In [22]:
gpt, ckpt = load_model("./dpo_stage1.pt")
gpt.to(device).train()
optimizer = torch.optim.AdamW(gpt.parameters(), lr=targeted_lr, weight_decay=0.01)
steps_per_epoch = len(targeted_lines) // targeted_batch_size
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=targeted_epochs * steps_per_epoch, eta_min=targeted_lr * 0.1)

In [23]:
ckpt_path = f"./dpo.pt"
total_steps = len(targeted_lines) // targeted_batch_size
for epoch in range(targeted_epochs):
    pbar = tqdm(get_batches(targeted_lines, targeted_batch_size))
    for step, (neg_tensor,pos_tensor) in enumerate(pbar):
        ###########################################################
        neg_logprob = compute_logprob(neg_tensor)
        pos_logprob = compute_logprob(pos_tensor)
        # preference term pushes the good answer above the refusal;
        # the extra -0.1*pos term also raises the good answer's likelihood directly
        loss = -F.logsigmoid((pos_logprob - neg_logprob) / targeted_beta).mean() - pos_logprob.mean() * 0.1
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(gpt.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        pbar.set_description(f"epoch {epoch} loss {loss.item():.4f} "
                             f"pos {pos_logprob.mean().item():.3f} neg {neg_logprob.mean().item():.3f}")
        ###########################################################
    torch.save({
        "model_state_dict": gpt.state_dict(),
        "model_args": ckpt['model_args'],
    }, ckpt_path)
    print(f"Saved checkpoint to {ckpt_path}")

epoch 0 loss 0.0168 pos -0.156 neg -9.017: : 1729it [02:34, 11.16it/s] 


Saved checkpoint to ./dpo.pt


epoch 1 loss 0.0161 pos -0.161 neg -9.731: : 1729it [02:36, 11.08it/s] 


Saved checkpoint to ./dpo.pt


epoch 2 loss 0.0153 pos -0.152 neg -9.872: : 1729it [02:36, 11.07it/s] 


Saved checkpoint to ./dpo.pt


... (10 outputs left out)

epoch 8 loss 0.0158 pos -0.158 neg -11.012: : 1729it [02:36, 11.06it/s]


Saved checkpoint to ./dpo.pt


epoch 9 loss 0.0155 pos -0.155 neg -11.089: : 1729it [02:36, 11.06it/s]


Saved checkpoint to ./dpo.pt


# Step 10A: Testing after targeted training

In [24]:
gpt, checkpoint = load_model("../dpo/dpo.pt")

In [25]:
# Test
gpt.eval()
test_set = ["17+19=?", "3*17=?", "72/4=?", "72-x=34,x=?", "x*11=44,x=?", "3*17=?", "72/4=?", "72-x=34,x=?"]
with torch.no_grad():
    for prompt in test_set:
        prompt_ids = encode(prompt)
        ###########################################################
        x = torch.tensor(prompt_ids, dtype=torch.long, device=device)[None, ...]
        y, _ = gpt.generate(x, max_new_tokens, temperature=temperature, top_k=top_k)
        print(decode(y[0].tolist()))
        ###########################################################

17+19=? The answer is 36 because 17+19 equals 36.
3*17=? The answer is 51 because 3*17 equals 51.
72/4=? The answer is 18 because 72/4 equals 18.
72-x=34,x=? The answer is 38 because 72-34 equals to 38.
x*11=44,x=? The answer is 4 because 44/11 equals to 4.
3*17=? The answer is 51 because 3*17 equals 51.
72/4=? The answer is 18 because 72/4 equals 18.
72-x=34,x=? The answer is 38 because 72-34 equals to 38.


# Step 10B: Additional evaluation after targeted training

In [26]:
model, _ = load_model("./dpo.pt")
acc, targeted_pairs = evaluate_model(model, "./pos_neg_pairs.json", build_targeted=False, num_samples=None)

evaluating: 100%|██████████| 33733/33733 [2:01:42<00:00,  4.62it/s]  

correct: 33732/33733 = 100.00%


# Step 10C: Out of Distribution Test Set

In [27]:
# Test
gpt.eval()
test_set = ["17+100=?", "x*0=0,x=?", "2-5=?", "50+50=?"]
with torch.no_grad():
    for prompt in test_set:
        prompt_ids = encode(prompt)
        ###########################################################
        x = torch.tensor(prompt_ids, dtype=torch.long, device=device)[None, ...]
        y, _ = gpt.generate(x, max_new_tokens, temperature=temperature, top_k=top_k)
        print(decode(y[0].tolist()))
        ###########################################################

17+100=? The answer is 1 because 10+10 equals 11.
x*0=0,x=? The answer is 0 because 0/0 equals to 0.
2-5=? The answer is 7 because 2-5 equals 7.
50+50=? The answer is 0 because 50+50 equals 0.


# Learning, Observation and Reflection

<u><b>Dataset Generation</b></u>
1. It is important to study and understand the data before working on generating it, in our case, we realised that the data is simple enough that it can be generated programmatically instead of using an LLM, this give us several benefits:
    - Firstly, eliminate unnecessary overhead of LLM
    - More control over the data generated (rules etc.)
2. This ties well to the second part of the course about knowledge representation, here we can see how our knowledge on the data allow us to create explicit rules which in turn help to improve the model.

<u><b>Model Training and Behavior</b></u>
1. The model is quite data hungry, even though we provided the same data, if we don't give at least 100k samples, it does not learn well, as discussed in our rationale for duplicating the data
2. The model is very susceptible to noise and does not learn well when that happens, in our case, our data generation allowed us to introduce some noise which seems quite minimal, but doing so hampers the model ability to learn. In the end, we trained the model with only the "Not Sure" negative examples.
3. The first time we got all test examples correct was with an epoch of 10, but this was inconsistent so we increased it to 15.
4. The loss indicated during the training process refers to the training loss and is not a good indicator of the model performance, a drop in loss can show that the model is learning properly but a lower loss does not necessarily translate to a better performance.

<u><b>Targeted Training</b></u>
1. The dataset used for targeted training have to include a large amount of data from the original dataset. We initially used only the targeted pairs and duplicate them to reach 100k samples, this resulted in extremely high losses even though the model has already learned well from the initial training.
2. Targeted training seems to help (improving the model performance from 99.943% to 99.997%), this is not a big improvement by percentage alone but the model already peforms very well, so a big improvement is unrealistic. If we look at the training loss, the model training on the original dataset plateau around 0.165 but with the added targeted training, the loss is able to go beyond that point to around 0.155, and unlike before where using anything other than "Not Sure" gives a high loss, with the combined targeted pairs and original pairs, this is no longer a problem and the losses is actually lower than just the original dataset, suggesting that targeted training improves the model.
    - Additionally, the improvement were consistent over many runs, the numbers differ but in every run, targeted training improved the model, the highest improvement we have got was with the initial training running for 10 epochs, targeted training improved the loss from around 0.160 to around 0.130, resulting in an accuracy increase of 99.5% to 99.9%.
    - However we can't conclude conclusively that targeted training improves the model because it may just be because the dataset given is bigger or more training.

<u><b>Limitations</b></u>
1. We did not have a separate training and testing set. We did not split our dataset as it is already quite small, and overfitting is less of a concern here because the <u>scope of the problem(based off our assumptions of what are the possible prompts)</u> is small and our generation include every possible example so having separate split would likely only hurt the performance and make it less optimized for the assessment. However, if the test set used to assess our model is completely different from what we expect, then this would definitely hurt the model performance due to our model inability to generalise, as seen in the Out of Distribution tests. As far as evaluation is concerned, our reported metrics use the same training split.

<u><b>What could be better?</b></u>
1. Right now, we just take the last epoch but this may not be the best model, as seen in the loss. It might be better if we take the checkpoint with the best loss instead.
2. It would be nice to try having a train-test split too.

# Thank you for reading our project ! 😊